In [110]:
%load_ext autoreload
%autoreload 2
from dig4bio.io import read_raman_file
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from dig4bio.datasets import make_interim_transfer_plate_data

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [111]:
models = ['anton532','anton785','kaiser','metrohm','mettler','tec','timegate','tornado']

raw_by_model =  {model: read_raman_file(name=model,level='raw') for model in models}

In [157]:
print('*** Columns ***')
for model in raw_by_model.keys():
    print(f'{model}: {raw_by_model[model].columns[0:4].tolist() + ['...'] + raw_by_model[model].columns[-8:].tolist()}')

*** Columns ***
anton532: ['200.0', '202.0', '204.0', '206.0', '...', '3496.0', '3498.0', '3500.0', 'glucose', 'Na_acetate', 'Mg_SO4', 'MSM_present', 'fold_idx']
anton785: ['100.0', '102.0', '104.0', '106.0', '...', '2296.0', '2298.0', '2300.0', 'glucose', 'Na_acetate', 'Mg_SO4', 'MSM_present', 'fold_idx']
kaiser: ['-36.3', '-36.0', '-35.7', '-35.4', '...', '1940.7001', '1941.0001', '1941.3001', 'glucose', 'Na_acetate', 'Mg_SO4', 'MSM_present', 'fold_idx']
metrohm: ['202.22', '204.59', '206.96', '209.32', '...', '3347.17', '3348.28', '3349.39', 'glucose', 'Na_acetate', 'Mg_SO4', 'MSM_present', 'fold_idx']
mettler: ['300.0', '301.0', '302.0', '303.0', '...', '3198.0', '3199.0', '3200.0', 'glucose', 'Na_acetate', 'Mg_SO4', 'MSM_present', 'fold_idx']
tec: ['85.0', '86.0', '87.0', '88.0', '...', '3208.0', '3209.0', '3210.0', 'glucose', 'Na_acetate', 'Mg_SO4', 'MSM_present', 'fold_idx']
timegate: ['200.925569', '205.066898', '209.205702', '213.34198', '...', '1992.023478', '1994.858351', '1

In [112]:
def get_wavenumbers(df:pd.DataFrame):
    cols_to_numeric = pd.to_numeric(pd.Series(np.array(df.columns)),errors='coerce')
    return cols_to_numeric[cols_to_numeric.notna()].to_numpy()

wavenumbers = {}
summary_stats={}

for model,df in raw_by_model.items():

    numeric_cols = pd.to_numeric(df.columns, errors="coerce") # type: ignore
    spectral_cols = df.columns[numeric_cols.notna()]
    model_wavenumbers = numeric_cols[numeric_cols.notna()].astype(float)

    spacing = np.diff(model_wavenumbers)

    wavenumbers[model] =  model_wavenumbers.to_numpy()
    summary_stats[model] = {
            'measurement_count': df.shape[0],
            'max_intensity': np.max(df[spectral_cols]),
            'min_intensity': np.min(df[spectral_cols]),
            'mean_intensity': np.mean(df[spectral_cols]),
            'min_wavenumber': model_wavenumbers.min(),
            'max_wavenumber': model_wavenumbers.max(),
            'wavenumber_counts': len(model_wavenumbers),
            "min_wn_spacing": spacing.min(),
            "max_wn_spacing": spacing.max(),
            "median_wn_spacing": np.median(spacing),
            "unique_spacing": np.unique(np.round(spacing, 4)),
            "roughly_evenly_spaced": np.allclose(spacing, np.median(spacing), atol=10**-4),
            'Memory Usage (Mb)': round(df.memory_usage().sum()*10**-6,2),
            'Null Count': np.sum(df.isna())
            }

summary_stats_df = pd.DataFrame(summary_stats).transpose()
summary_stats_df

,measurement_count,max_intensity,min_intensity,mean_intensity,min_wavenumber,max_wavenumber,wavenumber_counts,min_wn_spacing,max_wn_spacing,median_wn_spacing,unique_spacing,roughly_evenly_spaced,Memory Usage (Mb),Null Count
anton532,270,12267.88,1009.39,2197.199728,200.0,3500.0,1651,2.0,2.0,2.0,[2.0],True,3.58,0
anton785,270,6985.63,443.91,844.658705,100.0,2300.0,1101,2.0,2.0,2.0,[2.0],True,2.39,0
kaiser,134,11191.9518,2.5966,2400.279402,-36.3,1941.3001,6593,0.3,0.3001,0.3,"[0.3, 0.3001]",True,7.07,0
metrohm,399,25064.0,1057.0,2173.048186,202.22,3349.39,1917,1.1,2.37,1.6,"[1.1, 1.11, 1.12, 1.13, 1.14, 1.15, 1.16, 1.17...",False,6.14,0
mettler,275,54678.70571,99.111623,5268.838992,300.0,3200.0,2901,1.0,1.0,1.0,[1.0],True,6.39,0
tec,395,43539.529994,0.0,5092.860207,85.0,3210.0,3126,1.0,1.0,1.0,[1.0],True,9.89,0
timegate,133,0.005121,0.000043,0.000568,200.925569,1997.690789,511,1.918722,7.138847,3.497699,"[1.9187, 2.8324, 2.8349, 2.8373, 2.8397, 2.842...",False,0.55,0
tornado,385,91506.351462,251.38228,10174.176162,300.0,3300.0,3001,1.0,1.0,1.0,[1.0],True,9.26,0


In [113]:
print('** Wavenumber Ranges **')
name_width = max(len(model) for model in summary_stats)
for model,stats in summary_stats.items():
    print(f'{model:<{name_width}}: {stats['min_wavenumber']:>8.2f} - {stats['max_wavenumber']:>7.2f}')

** Wavenumber Ranges **
anton532:   200.00 - 3500.00
anton785:   100.00 - 2300.00
kaiser  :   -36.30 - 1941.30
metrohm :   202.22 - 3349.39
mettler :   300.00 - 3200.00
tec     :    85.00 - 3210.00
timegate:   200.93 - 1997.69
tornado :   300.00 - 3300.00


In [175]:
## What is fold_idx?

print('fold_idx row counts')
for model,df in raw_by_model.items():
    print(f'{model}: {df.fold_idx.value_counts().to_dict()}')

fold_idx row counts
anton532: {1: 55, 2: 55, 3: 55, 4: 55, 0: 50}
anton785: {1: 55, 2: 55, 3: 55, 4: 55, 0: 50}
kaiser: {1: 27, 2: 27, 3: 27, 4: 27, 0: 26}
metrohm: {0: 80, 1: 80, 2: 80, 3: 80, 4: 79}
mettler: {0: 55, 1: 55, 2: 55, 3: 55, 4: 55}
tec: {1: 80, 2: 80, 3: 80, 4: 80, 0: 75}
timegate: {1: 27, 3: 27, 4: 27, 0: 26, 2: 26}
tornado: {2: 80, 4: 80, 0: 75, 1: 75, 3: 75}


In [159]:
raw_transfer_df = read_raman_file(name='transfer_plate',level='raw')

raw_transfer_df[raw_transfer_df.columns[0]] == raw_transfer_df[raw_transfer_df.columns[-4]]
raw_transfer_df.head()

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 2043,Unnamed: 2044,Unnamed: 2045,Unnamed: 2046,Unnamed: 2047,Unnamed: 2048,Analyte concentration,Glucose (g/L),Sodium Acetate (g/L),Magnesium Acetate (g/L)
0,sample1,[6293,7095,8325,9934,11917,14394,18925,34874,65535,...,1616,1024,1013,1067,1277,5618],sample1,4.619282,1.937172,1.052928
1,NaN,[6505,7332,8482,10175,12132,14792,19594,35813,65535,...,1655,1004,1032,1049,1271,5756],sample2,5.782718,1.175902,1.214738
2,sample2,[6478,7158,8444,9979,11932,14503,19309,35118,65535,...,1651,1024,1009,1049,1275,5685],sample3,3.953448,1.350473,2.132459
3,NaN,[6511,7308,8520,10205,12260,14777,19569,35825,65535,...,1623,1021,1008,1026,1250,5839],sample4,2.038084,0.948045,1.380240
4,sample3,[6561,7342,8562,10166,12202,14838,19593,35869,65535,...,1638,1010,1012,1047,1307,5801],sample5,4.978295,0.459765,2.539622


In [115]:
interim_transfer_df = make_interim_transfer_plate_data()
interim_transfer_df

,sample,65.0,66.6,68.21,69.81,71.42,73.02,74.63,76.23,77.84,...,3340.37,3341.98,3343.58,3345.19,3346.79,3348.4,3350.0,Glucose (g/L),Sodium Acetate (g/L),Magnesium Acetate (g/L)
0,sample1,6293,7095,8325,9934,11917,14394,18925,34874,65535,...,1603,1616,1024,1013,1067,1277,5618,4.619282,1.937172,1.052928
1,sample1,6505,7332,8482,10175,12132,14792,19594,35813,65535,...,1626,1655,1004,1032,1049,1271,5756,4.619282,1.937172,1.052928
2,sample2,6478,7158,8444,9979,11932,14503,19309,35118,65535,...,1615,1651,1024,1009,1049,1275,5685,5.782718,1.175902,1.214738
3,sample2,6511,7308,8520,10205,12260,14777,19569,35825,65535,...,1618,1623,1021,1008,1026,1250,5839,5.782718,1.175902,1.214738
4,sample3,6561,7342,8562,10166,12202,14838,19593,35869,65535,...,1601,1638,1010,1012,1047,1307,5801,3.953448,1.350473,2.132459
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,sample94,6652,7453,8641,10270,12168,15014,20000,36732,65535,...,1620,1619,1026,1025,1078,1294,5834,6.531089,1.762051,1.429380
188,sample95,6798,7514,8786,10431,12372,15419,20547,37854,65535,...,1649,1587,1035,1033,1063,1271,5985,3.369372,1.400029,2.647925
189,sample95,6764,7534,8828,10532,12454,15504,20566,37705,65535,...,1632,1640,1030,1006,1073,1280,5925,3.369372,1.400029,2.647925
190,sample96,6847,7545,8795,10452,12588,15515,20492,37710,65535,...,1655,1667,1052,1019,1048,1266,6056,5.893708,1.898131,1.406329


In [116]:
numeric_cols = pd.to_numeric(interim_transfer_df.columns, errors="coerce")  # type: ignore
spectral_cols = interim_transfer_df.columns[1:-4]
model_wavenumbers = numeric_cols[numeric_cols.notna()].astype(float)
spectral_values = interim_transfer_df[spectral_cols].apply(pd.to_numeric, errors="coerce")

spacing = np.diff(model_wavenumbers)

print("**Transfer Plate**")
print(f"Min Wavenumber: {model_wavenumbers.min()}")
print(f"Max Wavenumber: {model_wavenumbers.max()}")
print(f"Median Wavenumber Spacing: {np.median(spacing)}")
print(f"Roughly Evenly Spaced: {np.allclose(spacing, np.median(spacing), atol=10**-4)}")
print(f"Max Intensity: {spectral_values.to_numpy().max()}")
print(f"Min Intensity: {spectral_values.to_numpy().min()}")
print(f"Mean Intensity: {spectral_values.to_numpy().mean()}")
print(f'Any nulls in Interim Transfer Plate Data: {bool((interim_transfer_df.isna().sum()>0).any())}')

**Transfer Plate**
Min Wavenumber: 65.0
Max Wavenumber: 3350.0
Median Wavenumber Spacing: 1.6000000000003638
Roughly Evenly Spaced: False
Max Intensity: 65535
Min Intensity: 987
Mean Intensity: 4333.7842828936655
Any nulls in Interim Transfer Plate Data: False


In [124]:
target_samples_df = read_raman_file('96_samples',level='raw', header=None)
target_samples_df.head()

,0,1,2,3,4,5,6,7,8,9,...,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048
0,sample1,6409,7097,8247,9853,11768,14434,19349,36210,65535,...,1709,1706,1598,1627,1653,1008,1035,1061,1282,5611]
1,NaN,6445,7147,8385,9837,11930,14613,19526,37225,65535,...,1668,1659,1594,1625,1656,1022,1020,1057,1301,5731]
2,sample2,5749,6351,7447,8777,10491,12729,17102,32323,65535,...,1650,1617,1544,1558,1617,1012,1012,1059,1295,5112]
3,NaN,5888,6475,7461,8822,10565,12960,17295,32686,65535,...,1629,1644,1551,1545,1559,1013,1036,1070,1274,5161]
4,sample3,6247,6975,8166,9554,11401,14010,18883,35979,65535,...,1687,1705,1602,1634,1620,1008,1016,1051,1278,5513]
